In [4]:
# Print the exact column names of your fact_nav table
print("Columns in fact_nav table:")
print(fact_nav.columns.tolist())

Columns in fact_nav table:
['amfi_code', 'date', 'nav']


In [5]:
import pandas as pd
import numpy as np
import sqlite3
import plotly.express as px
from pathlib import Path

# 1. Setup the safe path to the main directory database
db_path = Path("..") / "bluestock_mf.db"
conn = sqlite3.connect(db_path)

# 2. Extract data into Pandas DataFrames
fact_nav = pd.read_sql_query("SELECT * FROM fact_nav", conn)
dim_fund = pd.read_sql_query("SELECT * FROM dim_fund", conn)
conn.close()

# 3. Process dates and sort data chronologically by scheme and date
fact_nav['date'] = pd.to_datetime(fact_nav['date'])
fact_nav = fact_nav.sort_values(by=['amfi_code', 'date']).reset_index(drop=True)

# 4. Compute daily returns grouped by each individual fund scheme (amfi_code)
# Formula: (NAV_t / NAV_t-1) - 1
fact_nav['daily_return'] = fact_nav.groupby('amfi_code')['nav'].pct_change()

print("--- Data Processing Complete ---")
print(fact_nav[['amfi_code', 'date', 'nav', 'daily_return']].dropna().head())

# 5. Filter out NaN values from the first operational day of each fund
clean_returns = fact_nav.dropna(subset=['daily_return'])

# 6. Generate the interactive Plotly distribution histogram
fig = px.histogram(
    clean_returns, 
    x="daily_return", 
    nbins=120,
    title="Distribution of Daily Returns across All Schemes",
    labels={'daily_return': 'Daily Return (% Change)'},
    marginal="box"
)

fig.update_layout(
    bargap=0.05, 
    title_x=0.5,
    xaxis_title="Daily Return Range",
    yaxis_title="Frequency Count"
)

# Display the interactive validation chart
fig.show()

--- Data Processing Complete ---
   amfi_code       date       nav  daily_return
1     100016 2022-01-04  515.0971     -0.010306
2     100016 2022-01-05  521.7239      0.012865
3     100016 2022-01-06  515.7880     -0.011377
4     100016 2022-01-07  515.1639     -0.001210
5     100016 2022-01-10  510.7136     -0.008639


In [6]:
# Create a new cell for CAGR Calculations
print("Calculating CAGR for 1Yr, 3Yr, and 5Yr horizons...")

# Get the latest date available in our dataset to anchor our time windows
latest_date = fact_nav['date'].max()
print(f"Latest anchor date in dataset: {latest_date.strftime('%Y-%m-%d')}")

# Initialize a list to hold records for our comparison table
cagr_records = []

# Group by each mutual fund scheme
for amfi, group in fact_nav.groupby('amfi_code'):
    group = group.sort_values('date')
    
    # Extract the absolute absolute latest NAV entry for this specific fund
    latest_row = group.iloc[-1]
    nav_end = latest_row['nav']
    date_end = latest_row['date']
    
    # Dictionary to store calculated CAGRs for this fund
    fund_cagr = {'amfi_code': amfi}
    
    # Calculate CAGR for each specified target window: 1 year, 3 years, 5 years
    for years in [1, 3, 5]:
        target_start_date = date_end - pd.DateOffset(years=years)
        
        # Find the closest available actual trading date on or right after the target start date
        available_data_window = group[group['date'] >= target_start_date]
        
        if not available_data_window.empty:
            start_row = available_data_window.iloc[0]
            nav_start = start_row['nav']
            date_start = start_row['date']
            
            # Compute precise delta time horizon 'n' in years
            n = (date_end - date_start).days / 365.25
            
            # Avoid division by zero or negative time spans
            if n > 0.9: 
                cagr_val = (nav_end / nav_start) ** (1 / n) - 1
                fund_cagr[f'{years}yr_cagr'] = cagr_val
            else:
                fund_cagr[f'{years}yr_cagr'] = np.nan
        else:
            fund_cagr[f'{years}yr_cagr'] = np.nan
            
    cagr_records.append(fund_cagr)

# Convert records into our final comparison table DataFrame
df_cagr = pd.DataFrame(cagr_records)

# Fetch fund names from dim_fund to make the comparison table human-readable
if 'scheme_name' in dim_fund.columns:
    df_cagr = df_cagr.merge(dim_fund[['amfi_code', 'scheme_name']], on='amfi_code', how='left')
elif 'fund_name' in dim_fund.columns:
    df_cagr = df_cagr.merge(dim_fund[['amfi_code', 'fund_name']], on='amfi_code', how='left')

# Reorder columns to place name up front
cols = ['amfi_code'] + ([c for c in df_cagr.columns if c in ['scheme_name', 'fund_name']]) + ['1yr_cagr', '3yr_cagr', '5yr_cagr']
df_cagr = df_cagr[[c for c in cols if c in df_cagr.columns]]

print("\n--- CAGR Comparison Table Across All Funds (Sample Preview) ---")
# Format display as percentages for clean reading
pd.set_option('display.float_format', lambda x: '{:.2%}'.format(x) if not pd.isna(x) else '-')
print(df_cagr.head(10))

Calculating CAGR for 1Yr, 3Yr, and 5Yr horizons...
Latest anchor date in dataset: 2026-05-29

--- CAGR Comparison Table Across All Funds (Sample Preview) ---
   amfi_code                                        scheme_name  1yr_cagr  \
0     100016          HDFC Top 100 Fund - Regular Plan - Growth    -2.23%   
1     100025       HDFC Short Term Debt Fund - Regular - Growth     3.71%   
2     100033  HDFC Mid-Cap Opportunities Fund - Regular - Gr...    53.28%   
3     101206      ABSL Frontline Equity Fund - Regular - Growth    47.96%   
4     101207             ABSL Small Cap Fund - Regular - Growth   -24.00%   
5     101208                ABSL Liquid Fund - Regular - Growth     7.24%   
6     102885         UTI Nifty 50 Index Fund - Regular - Growth    20.22%   
7     102886                UTI Mid Cap Fund - Regular - Growth   -16.81%   
8     102887              UTI Flexi Cap Fund - Regular - Growth    13.59%   
9     118632     Nippon India Large Cap Fund - Regular - Growth    34.01

In [8]:
# Create a new cell for Sharpe and Sortino Ratios
print("Calculating Sharpe and Sortino Ratios (Rf = 6.5%)...")

# Define annual risk-free rate and convert to daily equivalent
rf_annual = 0.065
rf_daily = rf_annual / 252

ratio_records = []

# Group by each scheme to calculate metrics on their daily returns
for amfi, group in fact_nav.dropna(subset=['daily_return']).groupby('amfi_code'):
    returns = group['daily_return']
    
    if len(returns) > 1:
        # 1. Total Volatility (Standard Deviation)
        total_std_daily = returns.std()
        
        # 2. Downside Volatility (Negative returns only)
        negative_returns = returns[returns < 0]
        downside_std_daily = negative_returns.std() if len(negative_returns) > 1 else total_std_daily
        
        # 3. Mean daily return
        mean_return_daily = returns.mean()
        
        # 4. Calculate Annualized Sharpe Ratio
        if total_std_daily > 0:
            sharpe = ((mean_return_daily - rf_daily) / total_std_daily) * np.sqrt(252)
        else:
            sharpe = np.nan
            
        # 5. Calculate Annualized Sortino Ratio
        if downside_std_daily > 0:
            sortino = ((mean_return_daily - rf_daily) / downside_std_daily) * np.sqrt(252)
        else:
            sortino = np.nan
    else:
        sharpe = np.nan
        sortino = np.nan
        
    ratio_records.append({
        'amfi_code': amfi,
        'sharpe_ratio': sharpe,
        'sortino_ratio': sortino
    })

# Convert to DataFrame
df_ratios = pd.DataFrame(ratio_records)

# Merge back with our existing df_cagr to start building our master dashboard
df_performance = pd.merge(df_cagr, df_ratios, on='amfi_code', how='left')

# Add performance ranking based on the Sharpe Ratio
df_performance['sharpe_rank'] = df_performance['sharpe_ratio'].rank(ascending=False, method='min')

# Clean formatting style helper for final presentation string display
df_display = df_performance.copy()
df_display['1yr_cagr'] = df_display['1yr_cagr'].apply(lambda x: f"{x:.2%}" if pd.notna(x) else "-")
df_display['3yr_cagr'] = df_display['3yr_cagr'].apply(lambda x: f"{x:.2%}" if pd.notna(x) else "-")
df_display['5yr_cagr'] = df_display['5yr_cagr'].apply(lambda x: f"{x:.2%}" if pd.notna(x) else "-")
df_display['sharpe_ratio'] = df_display['sharpe_ratio'].apply(lambda x: f"{x:.4f}" if pd.notna(x) else "-")
df_display['sortino_ratio'] = df_display['sortino_ratio'].apply(lambda x: f"{x:.4f}" if pd.notna(x) else "-")

print("\n--- Performance Analytics Dashboard (Sample Preview) ---")
print(df_display[['amfi_code', 'scheme_name', '3yr_cagr', 'sharpe_ratio', 'sortino_ratio', 'sharpe_rank']].head(10))

Calculating Sharpe and Sortino Ratios (Rf = 6.5%)...

--- Performance Analytics Dashboard (Sample Preview) ---
   amfi_code                                        scheme_name 3yr_cagr  \
0     100016          HDFC Top 100 Fund - Regular Plan - Growth    1.29%   
1     100025       HDFC Short Term Debt Fund - Regular - Growth    3.92%   
2     100033  HDFC Mid-Cap Opportunities Fund - Regular - Gr...   32.43%   
3     101206      ABSL Frontline Equity Fund - Regular - Growth   28.96%   
4     101207             ABSL Small Cap Fund - Regular - Growth   -4.15%   
5     101208                ABSL Liquid Fund - Regular - Growth    6.31%   
6     102885         UTI Nifty 50 Index Fund - Regular - Growth   19.66%   
7     102886                UTI Mid Cap Fund - Regular - Growth   -0.77%   
8     102887              UTI Flexi Cap Fund - Regular - Growth   25.55%   
9     118632     Nippon India Large Cap Fund - Regular - Growth   22.65%   

  sharpe_ratio sortino_ratio  sharpe_rank  
0      -

In [10]:
import pandas as pd
import numpy as np
import sqlite3
import scipy.stats as stats
from pathlib import Path

print("Calculating Alpha and Beta against Nifty 100...")

# 1. Open connection to pull benchmark data
conn = sqlite3.connect(Path("..") / "bluestock_mf.db")

# Read the full fact_nav table to find the benchmark data rows
df_all_nav = pd.read_sql_query("SELECT * FROM fact_nav", conn)
conn.close()

df_all_nav['date'] = pd.to_datetime(df_all_nav['date'])

# 2. Extract Nifty 100 benchmark rows by searching for common names/codes
# If you have a specific code like '100016', replace the search string below
df_benchmark = df_all_nav[df_all_nav['amfi_code'].astype(str).str.contains('Nifty|NIFTY|100016', na=False, case=False)].copy()

if df_benchmark.empty:
    # If no explicit benchmark row is found, use the first fund in the database as a proxy market baseline
    first_fund = df_all_nav['amfi_code'].iloc[0]
    df_benchmark = df_all_nav[df_all_nav['amfi_code'] == first_fund].copy()

# 3. Calculate daily returns for the benchmark
df_benchmark = df_benchmark.sort_values('date').reset_index(drop=True)
df_benchmark['bench_return'] = df_benchmark['nav'].pct_change()
df_bench_clean = df_benchmark.dropna(subset=['bench_return'])[['date', 'bench_return']]

regression_records = []

# 4. Run OLS Linear Regression for each fund scheme
for amfi, group in fact_nav.dropna(subset=['daily_return']).groupby('amfi_code'):
    merged = pd.merge(group[['date', 'daily_return']], df_bench_clean, on='date', how='inner')
    
    if len(merged) > 15:
        # Run linear regression: market returns (X) vs fund returns (Y)
        beta, intercept, r_value, p_value, std_err = stats.linregress(merged['bench_return'], merged['daily_return'])
        alpha_annualized = intercept * 252  # Annualize the daily intercept
    else:
        beta = np.nan
        alpha_annualized = np.nan
        
    regression_records.append({
        'amfi_code': amfi,
        'beta': beta,
        'alpha': alpha_annualized
    })

# 5. Merge calculations back into our master dataframe matrix
df_reg = pd.DataFrame(regression_records)

# Check if alpha/beta columns exist from a previous bad run and drop them to prevent duplicates
columns_to_drop = [col for col in ['alpha', 'beta'] if col in df_performance.columns]
if columns_to_drop:
    df_performance = df_performance.drop(columns=columns_to_drop)

df_performance = pd.merge(df_performance, df_reg, on='amfi_code', how='left')

# 6. Safe formatting display function that won't crash pandas settings
df_display2 = df_performance.copy()
df_display2['3yr_cagr'] = df_display2['3yr_cagr'].apply(lambda x: f"{x:.2%}" if isinstance(x, float) and pd.notna(x) else str(x))
df_display2['sharpe_ratio'] = df_display2['sharpe_ratio'].apply(lambda x: f"{x:.4f}" if pd.notna(x) else "-")
df_display2['alpha'] = df_display2['alpha'].apply(lambda x: f"{x:.2%}" if pd.notna(x) else "-")
df_display2['beta'] = df_display2['beta'].apply(lambda x: f"{x:.4f}" if pd.notna(x) else "-")

print("\n--- Updated Performance Analytics Dashboard ---")
print(df_display2[['amfi_code', 'scheme_name', '3yr_cagr', 'sharpe_ratio', 'alpha', 'beta']].head(10))

Calculating Alpha and Beta against Nifty 100...

--- Updated Performance Analytics Dashboard ---
   amfi_code                                        scheme_name 3yr_cagr  \
0     100016          HDFC Top 100 Fund - Regular Plan - Growth    1.29%   
1     100025       HDFC Short Term Debt Fund - Regular - Growth    3.92%   
2     100033  HDFC Mid-Cap Opportunities Fund - Regular - Gr...   32.43%   
3     101206      ABSL Frontline Equity Fund - Regular - Growth   28.96%   
4     101207             ABSL Small Cap Fund - Regular - Growth   -4.15%   
5     101208                ABSL Liquid Fund - Regular - Growth    6.31%   
6     102885         UTI Nifty 50 Index Fund - Regular - Growth   19.66%   
7     102886                UTI Mid Cap Fund - Regular - Growth   -0.77%   
8     102887              UTI Flexi Cap Fund - Regular - Growth   25.55%   
9     118632     Nippon India Large Cap Fund - Regular - Growth   22.65%   

  sharpe_ratio   alpha     beta  
0      -0.2015   0.00%   1.0000 

In [11]:
# Create a new cell for Maximum Drawdown calculation
print("Computing Maximum Drawdown and Worst-Case Date Ranges...")

drawdown_records = []

# Loop through each fund scheme
for amfi, group in fact_nav.sort_values(['amfi_code', 'date']).groupby('amfi_code'):
    group = group.copy().reset_index(drop=True)
    
    # 1. Compute rolling cumulative peak value
    group['peak'] = group['nav'].cummax()
    
    # 2. Compute drawdown percentage drop
    group['drawdown'] = (group['nav'] / group['peak']) - 1
    
    # 3. Locate the absolute lowest drawdown point
    max_dd = group['drawdown'].min()
    
    if pd.notna(max_dd) and max_dd < 0:
        trough_idx = group['drawdown'].idxmin()
        trough_row = group.loc[trough_idx]
        trough_date = trough_row['date']
        
        # Look backwards from the trough to find the exact peak date that started the drop
        peak_val = trough_row['peak']
        peak_row = group.loc[:trough_idx][group.loc[:trough_idx]['nav'] == peak_val]
        peak_date = peak_row['date'].iloc[-1] if not peak_row.empty else group['date'].iloc[0]
    else:
        max_dd = 0.0
        peak_date = np.nan
        trough_date = np.nan
        
    drawdown_records.append({
        'amfi_code': amfi,
        'max_drawdown': max_dd,
        'drawdown_start_date': peak_date,
        'drawdown_trough_date': trough_date
    })

# Convert to DataFrame and merge with master performance stats
df_dd = pd.DataFrame(drawdown_records)

# Check and drop columns if they already exist to avoid duplication crashes
columns_to_drop = [c for c in ['max_drawdown', 'drawdown_start_date', 'drawdown_trough_date'] if c in df_performance.columns]
if columns_to_drop:
    df_performance = df_performance.drop(columns=columns_to_drop)

df_performance = pd.merge(df_performance, df_dd, on='amfi_code', how='left')

# 4. Safe presentation string display
df_display3 = df_performance.copy()
df_display3['max_drawdown'] = df_display3['max_drawdown'].apply(lambda x: f"{x:.2%}" if pd.notna(x) else "-")
df_display3['drawdown_start'] = df_display3['drawdown_start_date'].dt.strftime('%Y-%m-%d')
df_display3['drawdown_trough'] = df_display3['drawdown_trough_date'].dt.strftime('%Y-%m-%d')

print("\n--- Performance Analytics Dashboard with Max Drawdowns ---")
print(df_display3[['amfi_code', 'scheme_name', 'max_drawdown', 'drawdown_start', 'drawdown_trough']].head(10))

Computing Maximum Drawdown and Worst-Case Date Ranges...

--- Performance Analytics Dashboard with Max Drawdowns ---
   amfi_code                                        scheme_name max_drawdown  \
0     100016          HDFC Top 100 Fund - Regular Plan - Growth      -24.73%   
1     100025       HDFC Short Term Debt Fund - Regular - Growth       -4.31%   
2     100033  HDFC Mid-Cap Opportunities Fund - Regular - Gr...      -16.22%   
3     101206      ABSL Frontline Equity Fund - Regular - Growth      -11.29%   
4     101207             ABSL Small Cap Fund - Regular - Growth      -35.45%   
5     101208                ABSL Liquid Fund - Regular - Growth       -0.16%   
6     102885         UTI Nifty 50 Index Fund - Regular - Growth      -10.86%   
7     102886                UTI Mid Cap Fund - Regular - Growth      -28.00%   
8     102887              UTI Flexi Cap Fund - Regular - Growth      -21.54%   
9     118632     Nippon India Large Cap Fund - Regular - Growth      -17.41%   

  

In [12]:
# Create a new cell for the Final Composite Scorecard
print("Generating Final Fund Scorecard (0-100)...")

# 1. Ensure expense_ratio is attached to our performance table from dim_fund
if 'expense_ratio' not in df_performance.columns:
    # Check what columns are available to map expense_ratio safely
    expense_col = [c for c in dim_fund.columns if 'expense' in c.lower()]
    if expense_col:
        df_performance = pd.merge(
            df_performance, 
            dim_fund[['amfi_code', expense_col[0]]], 
            on='amfi_code', 
            how='left'
        ).rename(columns={expense_col[0]: 'expense_ratio'})
    else:
        # Fallback placeholder if no expense ratio column exists in the database
        df_performance['expense_ratio'] = 0.015 

# Fill any missing data placeholders to avoid rank corruption
df_performance['3yr_cagr'] = df_performance['3yr_cagr'].fillna(df_performance['3yr_cagr'].median())
df_performance['sharpe_ratio'] = df_performance['sharpe_ratio'].fillna(df_performance['sharpe_ratio'].median())
df_performance['alpha'] = df_performance['alpha'].fillna(0.0)
df_performance['expense_ratio'] = df_performance['expense_ratio'].fillna(0.02)
df_performance['max_drawdown'] = df_performance['max_drawdown'].fillna(df_performance['max_drawdown'].median())

# 2. Compute Percentage Ranks (Values mapped between 0.0 and 1.0)
# Higher is better metrics
df_performance['rank_return'] = df_performance['3yr_cagr'].rank(pct=True)
df_performance['rank_sharpe'] = df_performance['sharpe_ratio'].rank(pct=True)
df_performance['rank_alpha']  = df_performance['alpha'].rank(pct=True)

# Lower is better metrics (Inverse ranking)
df_performance['rank_expense'] = df_performance['expense_ratio'].rank(pct=True, ascending=False)
# Since Max DD is negative (e.g. -25%), ascending=True makes less negative (smaller drops) rank higher
df_performance['rank_drawdown'] = df_performance['max_drawdown'].rank(pct=True, ascending=True)

# 3. Calculate Weighted Composite Score (Scaled out of 100)
df_performance['composite_score'] = (
    (df_performance['rank_return']   * 0.30) +
    (df_performance['rank_sharpe']   * 0.25) +
    (df_performance['rank_alpha']    * 0.20) +
    (df_performance['rank_expense']  * 0.15) +
    (df_performance['rank_drawdown'] * 0.10)
) * 100

# 4. Determine Final Overall Ranking Placement
df_performance['final_rank'] = df_performance['composite_score'].rank(ascending=False, method='min')

# 5. Export to CSV Deliverable as requested by assignment
output_csv_path = Path("..") / "fund_scorecard.csv"
df_performance.to_csv(output_csv_path, index=False)
print(f"Success! Final scorecard metrics exported safely to {output_csv_path.resolve()}")

# 6. Clean formatting display for final analytical review
df_final_display = df_performance.sort_values(by='final_rank').copy()
df_final_display['3yr_cagr'] = df_final_display['3yr_cagr'].apply(lambda x: f"{x:.2%}")
df_final_display['sharpe_ratio'] = df_final_display['sharpe_ratio'].apply(lambda x: f"{x:.4f}")
df_final_display['max_drawdown'] = df_final_display['max_drawdown'].apply(lambda x: f"{x:.2%}")
df_final_display['composite_score'] = df_final_display['composite_score'].apply(lambda x: f"{x:.2f}")

print("\n🏆 --- TOP 10 MUTUAL FUNDS RANKED BY COMPOSITE SCORE --- 🏆")
print(df_final_display[['final_rank', 'amfi_code', 'scheme_name', '3yr_cagr', 'sharpe_ratio', 'max_drawdown', 'composite_score']].head(10))

Generating Final Fund Scorecard (0-100)...
Success! Final scorecard metrics exported safely to C:\Users\lenovo\OneDrive\Desktop\mutual_fund_explorer\fund_scorecard.csv

🏆 --- TOP 10 MUTUAL FUNDS RANKED BY COMPOSITE SCORE --- 🏆
    final_rank  amfi_code                                        scheme_name  \
34      1.0000     148567      Mirae Asset Large Cap Fund - Regular - Growth   
25      2.0000     120505           ICICI Pru Midcap Fund - Regular - Growth   
30      3.0000     120843             Kotak Flexicap Fund - Regular - Growth   
2       4.0000     100033  HDFC Mid-Cap Opportunities Fund - Regular - Gr...   
24      5.0000     120504          ICICI Pru Bluechip Fund - Direct - Growth   
16      6.0000     119094                Axis Midcap Fund - Regular - Growth   
19      7.0000     119551          SBI Bluechip Fund - Regular Plan - Growth   
36      8.0000     148569      Mirae Asset Tax Saver Fund - Regular - Growth   
3       9.0000     101206      ABSL Frontline Equity 

In [15]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import sqlite3
from pathlib import Path

print("Generating 3-Year Benchmark Comparison Chart & Tracking Error Analytics...")

# 1. Identify the Top 5 Fund AMFI codes based on final composite ranking
top_5_amfi = df_performance.sort_values(by='composite_score', ascending=False)['amfi_code'].head(5).tolist()

# 2. Re-extract daily NAV sequences over a 3-year historical lookback window
conn = sqlite3.connect(Path("..") / "bluestock_mf.db")
df_nav_raw = pd.read_sql_query("SELECT amfi_code, date, nav FROM fact_nav", conn)
conn.close()

df_nav_raw['date'] = pd.to_datetime(df_nav_raw['date'])
max_date = df_nav_raw['date'].max()
start_lookback = max_date - pd.DateOffset(years=3)

# Filter for the relevant 3-year window
df_3yr = df_nav_raw[(df_nav_raw['date'] >= start_lookback) & (df_nav_raw['date'] <= max_date)].copy()

# 3. Setup the benchmark line
df_bench_line = df_benchmark[(df_benchmark['date'] >= start_lookback) & (df_benchmark['date'] <= max_date)].copy()
df_bench_line = df_bench_line.sort_values('date').reset_index(drop=True)
df_bench_line['normalized_nav'] = (df_bench_line['nav'] / df_bench_line['nav'].iloc[0]) * 100
df_bench_line['daily_return'] = df_bench_line['nav'].pct_change()

# 4. Generate the interactive visualization tracking lines
fig_cum = go.Figure()

# Add the main market benchmark comparison line
fig_cum.add_trace(go.Scatter(
    x=df_bench_line['date'], 
    y=df_bench_line['normalized_nav'],
    name='Nifty Benchmark Baseline',
    line=dict(color='white', width=3, dash='dash')
))

tracking_errors = []

# Loop through and process each of the top 5 funds individually
for amfi in top_5_amfi:
    fund_data = df_3yr[df_3yr['amfi_code'] == amfi].sort_values('date').copy().reset_index(drop=True)
    if not fund_data.empty:
        name_match = df_performance[df_performance['amfi_code'] == amfi]['scheme_name'].values
        name = name_match[0] if len(name_match) > 0 else f"Fund {amfi}"
        short_name = name.split(" - ")[0]
        
        # Calculate Normalized growth path (Base 100)
        fund_data['normalized_nav'] = (fund_data['nav'] / fund_data['nav'].iloc[0]) * 100
        fund_data['daily_return'] = fund_data['nav'].pct_change()
        
        # Plot growth trail line
        fig_cum.add_trace(go.Scatter(
            x=fund_data['date'], 
            y=fund_data['normalized_nav'], 
            name=short_name
        ))
        
        # Compute Tracking Error against the market baseline index
        merged_returns = pd.merge(
            fund_data[['date', 'daily_return']], 
            df_bench_line[['date', 'daily_return']], 
            on='date', 
            suffixes=('_fund', '_bench')
        ).dropna()
        
        return_diff = merged_returns['daily_return_fund'] - merged_returns['daily_return_bench']
        te_val = return_diff.std() * np.sqrt(252)
        tracking_errors.append({'amfi_code': amfi, 'scheme_name': short_name, 'tracking_error': te_val})

# Configure chart layout rules
fig_cum.update_layout(
    title="3-Year Cumulative Performance Comparison (Top 5 Funds vs Benchmark)",
    title_x=0.5,
    xaxis_title="Timeline Horizon",
    yaxis_title="Growth Base Index Value (Normalized to 100)",
    template="plotly_dark",
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

# 5. Export alpha_beta.csv report safely as required by deliverables checklist
df_alpha_beta = df_performance[['amfi_code', 'scheme_name', 'alpha', 'beta']].copy()
df_alpha_beta.to_csv(Path("..") / "alpha_beta.csv", index=False)
print("Success! alpha_beta.csv file exported to root directory.")

# 6. Display Tracking error calculation statistics summary
df_te = pd.DataFrame(tracking_errors)
print("\n🎯 --- TOP 5 FUNDS ANNUALIZED TRACKING ERROR STATS --- 🎯")
df_te['tracking_error'] = df_te['tracking_error'].apply(lambda x: f"{x:.4%}")
print(df_te)

# Show graphic plot inside notebook workspace window
fig_cum.show()

Generating 3-Year Benchmark Comparison Chart & Tracking Error Analytics...
Success! alpha_beta.csv file exported to root directory.

🎯 --- TOP 5 FUNDS ANNUALIZED TRACKING ERROR STATS --- 🎯
   amfi_code                      scheme_name tracking_error
0     148567       Mirae Asset Large Cap Fund       19.6649%
1     120505            ICICI Pru Midcap Fund       24.8675%
2     120843              Kotak Flexicap Fund       21.3465%
3     100033  HDFC Mid-Cap Opportunities Fund       23.5853%
4     120504          ICICI Pru Bluechip Fund       20.9534%
